In [ ]:
from utils import train_load_datasets_resnet as tr
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
import torch

In [ ]:
flag = 'dermamnist-e'
color = True # Colors for the flags
batch_size = 128       # <- enable/disable RandAugment here
cuda = 'cuda:1'

device = torch.device(cuda if torch.cuda.is_available() else 'cpu')
size = 224  # Image size for the models

transform_eval = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[.5, .5, .5], std=[.5, .5, .5])
        ])

# Load plain datasets/loaders (no augmentation)
[study_dataset_plain, calibration_dataset, test_dataset], [_, calibration_loader, test_loader], info = tr.load_datasets(flag, color, size, transform_eval, batch_size)


In [ ]:
models = tr.load_models(flag, waugmentation=True, device=device)

In [ ]:
# Filter test samples whose center contains "external" and run ensemble predictions
external_indices = [i for i, c in enumerate(test_dataset.test_centers) if isinstance(c, str) and ("external" in c.lower())]

external_subset = torch.utils.data.Subset(test_dataset, external_indices)
external_loader = DataLoader(external_subset, batch_size=batch_size, shuffle=False)

# Ensure eval mode
for m in models:
    m.eval()

y_true_external = []
y_pred_external = []
y_prob_external = []

with torch.no_grad():
    # Collect per-model outputs to compute per-model metrics later
    per_model_probs = [[] for _ in models]  # list of lists of [B, C] (CPU)
    per_model_preds = [[] for _ in models]  # list of lists of [B] (CPU)

    for x, y in external_loader:
        x = x.to(device, non_blocking=True)

        probs_list = []
        for mi, m in enumerate(models):
            logits = m(x)
            probs = torch.softmax(logits, dim=1)
            probs_list.append(probs)

            # Save per-model outputs on CPU
            per_model_probs[mi].append(probs.cpu())
            per_model_preds[mi].append(probs.argmax(dim=1).cpu())

        probs_ens = torch.stack(probs_list, dim=0).mean(dim=0)  # [B, C]
        preds = probs_ens.argmax(dim=1)  # [B]

        y_prob_external.append(probs_ens.cpu())
        y_pred_external.append(preds.cpu())
        y_true_external.append(y.view(-1).cpu().long())

y_prob_external = torch.cat(y_prob_external, dim=0)
y_pred_external = torch.cat(y_pred_external, dim=0)
y_true_external = torch.cat(y_true_external, dim=0)

# Compute std of metrics over individual models
accs_external = []
baccs_external = []
aucs_external = []

num_classes_local = y_prob_external.shape[1]
for mi in range(len(models)):
    probs_i = torch.cat(per_model_probs[mi], dim=0)  # [N, C]
    preds_i = torch.cat(per_model_preds[mi], dim=0)  # [N]

    acc_i = (preds_i == y_true_external).float().mean().item()
    bacc_i = balanced_accuracy_score(y_true_external.numpy(), preds_i.numpy())
    try:
        auc_i = roc_auc_score(
            y_true_external.numpy(),
            probs_i.numpy(),
            labels=list(range(num_classes_local)),
            multi_class='ovr',
            average='macro',
        )
    except Exception:
        auc_i = float('nan')

    accs_external.append(acc_i)
    baccs_external.append(bacc_i)
    aucs_external.append(auc_i)

acc_std_external = torch.tensor(accs_external).std(unbiased=False).item()
bacc_std_external = torch.tensor(baccs_external).std(unbiased=False).item()
aucs_tensor = torch.tensor([a for a in aucs_external if a == a])  # drop NaNs
auc_std_external = (
    aucs_tensor.std(unbiased=False).item() if aucs_tensor.numel() > 0 else float('nan')
)

acc_external = (y_pred_external == y_true_external).float().mean().item()
# Balanced accuracy
bacc_external = balanced_accuracy_score(
    y_true_external.numpy(), y_pred_external.numpy()
)

# Macro AUC (OvR) across all classes; handle edge cases with try/except
num_classes = y_prob_external.shape[1]
try:
    auc_external = roc_auc_score(
        y_true_external.numpy(),
        y_prob_external.numpy(),
        labels=list(range(num_classes)),
        multi_class='ovr',
        average='macro',
    )
except Exception:
    auc_external = float('nan')

print(f"External test samples: {len(external_indices)}")
print(f"Ensemble accuracy on external subset: {acc_external:.4f}")
print(f"Balanced accuracy on external subset: {bacc_external:.4f}")
print(f"Macro AUC (OvR) on external subset: {auc_external:.4f}")
print(f"Std of individual model accuracies on external subset: {acc_std_external:.4f}")
print(f"Std of individual model balanced accuracies on external subset: {bacc_std_external:.4f}")
print(f"Std of individual model AUCs on external subset: {auc_std_external:.4f}")

# Variables produced:
# - external_indices: list of indices into test_dataset filtered by 'external'
# - external_subset: Subset of test_dataset with only external samples
# - external_loader: DataLoader over external_subset
# - y_true_external: tensor [N] of ground-truth labels (CPU)
# - y_pred_external: tensor [N] of predicted labels (CPU)
# - y_prob_external: tensor [N, num_classes] of ensemble probabilities (CPU)
# - acc_external: float accuracy on external subset